In [ ]:
# ----------------------------------------------------------
# NOTEBOOK: nb_mongodb_ingestion_bronze
# ----------------------------------------------------------
# PURPOSE
# -------
# This notebook demonstrates secure NoSQL ingestion from MongoDB Atlas
# into Microsoft Fabric Bronze layer.
#
# FINAL AUTHENTICATION APPROACH
# -----------------------------
# Azure Key Vault was implemented as the intended enterprise secret store.
# However, direct Key Vault retrieval from Fabric notebook execution was
# not usable in this environment because of:
# 1. runtime authentication limitation for DefaultAzureCredential()
# 2. tenant / issuer mismatch for direct Key Vault secret retrieval
#
# Therefore, Fabric Variable Library is used for runtime configuration.
# NotebookUtils reads the MongoDB settings at runtime without hardcoding
# credentials directly in notebook code.
#
# SOURCE
# ------
# MongoDB Atlas
# Database   : ecommerce_nosql_db
# Collection : orders_collection
#
# TARGET TABLES
# -------------
# bronze_mongodb_orders_raw
# bronze_mongodb_orders_flat

In [2]:
# ----------------------------------------------------------
# STEP 1: INSTALL REQUIRED LIBRARY
# ----------------------------------------------------------
# pymongo:
#   Python MongoDB client used to connect to MongoDB Atlas.

%pip install pymongo

StatementMeta(, b12ffaba-d39d-478c-a983-946771fd6bbd, 9, Finished, Available, Finished, False)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 16.5 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 331.1/331.1 kB 27.5 MB/s eta 0:00:00

[notice] A new release of pip is available: 24.0 -> 26.0.1
[notice] To update, run: python -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.



In [3]:
# ----------------------------------------------------------
# STEP 2: RESTART NOTE
# ----------------------------------------------------------
# If pymongo was just installed for the first time in this session,
# restart the session and then run the notebook again from the top.

print("If pymongo was just installed, restart the session and run again from the top.")

StatementMeta(, b12ffaba-d39d-478c-a983-946771fd6bbd, 11, Finished, Available, Finished, False)

If pymongo was just installed, restart the session and run again from the top.


In [4]:
# ----------------------------------------------------------
# STEP 3: IMPORT REQUIRED LIBRARIES
# ----------------------------------------------------------
# notebookutils:
#   Fabric-native utility used here to read Variable Library values.
#
# MongoClient:
#   Connects Python to MongoDB Atlas.
#
# col:
#   Used later to flatten nested JSON fields.

import notebookutils
from pymongo import MongoClient
from pyspark.sql.functions import col

print("Libraries imported successfully.")

StatementMeta(, b12ffaba-d39d-478c-a983-946771fd6bbd, 12, Finished, Available, Finished, False)

Libraries imported successfully.


In [5]:
# ----------------------------------------------------------
# STEP 4: FETCH MONGODB CONFIGURATION FROM FABRIC VARIABLE LIBRARY
# ----------------------------------------------------------
# Library name must match exactly, including case.
# Variables are read from the active value set.

mongodb_config = notebookutils.variableLibrary.getLibrary("mongodb_config")

mongo_username = mongodb_config.MONGO_USERNAME
mongo_password = mongodb_config.MONGO_PASSWORD
mongo_cluster_host = mongodb_config.MONGO_CLUSTER_HOST

if not mongo_username or not mongo_password or not mongo_cluster_host:
    raise ValueError(
        "MongoDB configuration values could not be read from Fabric Variable Library."
    )

print("MongoDB configuration fetched successfully from Fabric Variable Library.")
print("Username loaded:", mongo_username)
print("Cluster host loaded:", mongo_cluster_host)
print("Password loaded:", mongo_password is not None)

StatementMeta(, b12ffaba-d39d-478c-a983-946771fd6bbd, 13, Finished, Available, Finished, False)

MongoDB configuration fetched successfully from Fabric Variable Library.
Username loaded: santhoshsha_db_user
Cluster host loaded: cluster-ecom-nosql-dev.us2p7ir.mongodb.net
Password loaded: True


In [6]:
# ----------------------------------------------------------
# STEP 5: BUILD MONGODB CONNECTION STRING
# ----------------------------------------------------------
# Username, password, and cluster host are all retrieved from
# Fabric Variable Library.
#
# No credentials are hardcoded in notebook code.

connection_string = (
    f"mongodb+srv://{mongo_username}:{mongo_password}@{mongo_cluster_host}/"
    "?retryWrites=true&w=majority"
)

print("MongoDB connection string built securely.")

StatementMeta(, b12ffaba-d39d-478c-a983-946771fd6bbd, 14, Finished, Available, Finished, False)

MongoDB connection string built securely.


In [7]:
# ----------------------------------------------------------
# STEP 6: CONNECT TO MONGODB ATLAS
# ----------------------------------------------------------
# MongoClient():
#   Opens a connection to MongoDB Atlas.
#
# db:
#   Selects the MongoDB database.
#
# collection:
#   Selects the collection that contains the order documents.

mongo_client = MongoClient(connection_string)

db = mongo_client["ecommerce_nosql_db"]
collection = db["orders_collection"]

print("Connected to MongoDB Atlas successfully.")

StatementMeta(, b12ffaba-d39d-478c-a983-946771fd6bbd, 15, Finished, Available, Finished, False)

Connected to MongoDB Atlas successfully.


In [8]:
# ----------------------------------------------------------
# STEP 7: READ DOCUMENTS FROM MONGODB COLLECTION
# ----------------------------------------------------------
# collection.find():
#   Reads all documents from the selected collection.
#
# list(...):
#   Converts MongoDB cursor output into a Python list.

documents = list(collection.find())

print(f"Total MongoDB documents read: {len(documents)}")

if len(documents) > 0:
    print("Sample MongoDB document:")
    print(documents[0])
else:
    print("No documents found in collection.")

StatementMeta(, b12ffaba-d39d-478c-a983-946771fd6bbd, 16, Finished, Available, Finished, False)

Total MongoDB documents read: 1
Sample MongoDB document:
{'_id': ObjectId('69de5c58d5c58a217003c92e'), 'order_id': 1001, 'customer': {'customer_id': 301, 'name': 'Santhosh', 'email': 'santhosh@gmail.com'}, 'products': [{'product_id': 3001, 'product_name': 'Laptop', 'category': 'Electronics', 'price': 75000}, {'product_id': 3002, 'product_name': 'Mouse', 'category': 'Electronics', 'price': 500}], 'payment': {'payment_id': 6001, 'method': 'card', 'status': 'SUCCESS', 'amount': 75500}, 'order_status': 'DELIVERED', 'order_time': datetime.datetime(2026, 4, 14, 15, 30)}


In [9]:
# ----------------------------------------------------------
# STEP 8: CLEAN MONGODB _id FOR SPARK COMPATIBILITY
# ----------------------------------------------------------
# MongoDB automatically creates _id using ObjectId.
# Convert it to string so Spark can handle it cleanly.

for doc in documents:
    doc["_id"] = str(doc["_id"])

print("MongoDB _id converted to string successfully.")

StatementMeta(, b12ffaba-d39d-478c-a983-946771fd6bbd, 17, Finished, Available, Finished, False)

MongoDB _id converted to string successfully.


In [10]:
# ----------------------------------------------------------
# STEP 9: CONVERT DOCUMENTS TO SPARK DATAFRAME
# ----------------------------------------------------------
# spark.createDataFrame():
#   Converts Python list of dictionaries into a Spark DataFrame.

mongo_df_raw = spark.createDataFrame(documents)

print("Raw Spark DataFrame created from MongoDB documents.")
mongo_df_raw.printSchema()
display(mongo_df_raw)

StatementMeta(, b12ffaba-d39d-478c-a983-946771fd6bbd, 18, Finished, Available, Finished, False)

Raw Spark DataFrame created from MongoDB documents.
root
 |-- _id: string (nullable = true)
 |-- customer: map (nullable = true)
 |    |-- key: string
 |    |-- value: long (valueContainsNull = true)
 |-- order_id: long (nullable = true)
 |-- order_status: string (nullable = true)
 |-- order_time: timestamp (nullable = true)
 |-- payment: map (nullable = true)
 |    |-- key: string
 |    |-- value: long (valueContainsNull = true)
 |-- products: array (nullable = true)
 |    |-- element: map (containsNull = true)
 |    |    |-- key: string
 |    |    |-- value: long (valueContainsNull = true)



SynapseWidget(Synapse.DataFrame, 1d08ebd9-a14c-4ce0-a1b5-936c0dc39a40)

In [11]:
# ----------------------------------------------------------
# STEP 10: WRITE RAW MONGODB DATA TO BRONZE LAYER
# ----------------------------------------------------------
# This preserves the original nested NoSQL document structure.

mongo_df_raw.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("bronze_mongodb_orders_raw")

print("bronze_mongodb_orders_raw created successfully.")

StatementMeta(, b12ffaba-d39d-478c-a983-946771fd6bbd, 19, Finished, Available, Finished, False)

bronze_mongodb_orders_raw created successfully.


In [12]:
# ----------------------------------------------------------
# STEP 11: VALIDATE RAW BRONZE TABLE
# ----------------------------------------------------------

spark.sql("""
SELECT *
FROM bronze_mongodb_orders_raw
""").show(truncate=False)

StatementMeta(, b12ffaba-d39d-478c-a983-946771fd6bbd, 20, Finished, Available, Finished, False)

+------------------------+-------------------------------------------------+--------+------------+-------------------+---------------------------------------------------------------------+----------------------------------------------------------------------------------------------------------------------------------------------------------+
|_id                     |customer                                         |order_id|order_status|order_time         |payment                                                              |products                                                                                                                                                  |
+------------------------+-------------------------------------------------+--------+------------+-------------------+---------------------------------------------------------------------+----------------------------------------------------------------------------------------------------------------------------

In [13]:
# ----------------------------------------------------------
# STEP 12: FLATTEN NESTED MONGODB FIELDS
# ----------------------------------------------------------
# WHY THIS STEP MATTERS
# ---------------------
# MongoDB stores data in nested JSON-like documents.
# For analytics and downstream processing, flattening nested fields
# into top-level columns makes the data easier to query and join.
#
# We keep the products array for later advanced processing.

mongo_df_flat = mongo_df_raw.select(
    col("_id").alias("mongo_id"),
    col("order_id"),
    col("customer.customer_id").alias("customer_id"),
    col("customer.name").alias("customer_name"),
    col("customer.email").alias("customer_email"),
    col("products"),
    col("payment.payment_id").alias("payment_id"),
    col("payment.method").alias("payment_method"),
    col("payment.status").alias("payment_status"),
    col("payment.amount").alias("payment_amount"),
    col("order_status"),
    col("order_time")
)

print("Flattened MongoDB DataFrame created.")
mongo_df_flat.printSchema()
display(mongo_df_flat)

StatementMeta(, b12ffaba-d39d-478c-a983-946771fd6bbd, 21, Finished, Available, Finished, False)

Flattened MongoDB DataFrame created.
root
 |-- mongo_id: string (nullable = true)
 |-- order_id: long (nullable = true)
 |-- customer_id: long (nullable = true)
 |-- customer_name: long (nullable = true)
 |-- customer_email: long (nullable = true)
 |-- products: array (nullable = true)
 |    |-- element: map (containsNull = true)
 |    |    |-- key: string
 |    |    |-- value: long (valueContainsNull = true)
 |-- payment_id: long (nullable = true)
 |-- payment_method: long (nullable = true)
 |-- payment_status: long (nullable = true)
 |-- payment_amount: long (nullable = true)
 |-- order_status: string (nullable = true)
 |-- order_time: timestamp (nullable = true)



SynapseWidget(Synapse.DataFrame, e89028f8-80a8-4d77-ade4-c93f8e55ae8b)

In [14]:
# ----------------------------------------------------------
# STEP 13: WRITE FLATTENED OUTPUT TO BRONZE
# ----------------------------------------------------------
# This output is easier for joins, Silver transformations,
# and downstream reporting use cases.

mongo_df_flat.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("bronze_mongodb_orders_flat")

print("bronze_mongodb_orders_flat created successfully.")

StatementMeta(, b12ffaba-d39d-478c-a983-946771fd6bbd, 22, Finished, Available, Finished, False)

bronze_mongodb_orders_flat created successfully.


In [15]:
# ----------------------------------------------------------
# STEP 14: VALIDATE FLATTENED BRONZE TABLE
# ----------------------------------------------------------

spark.sql("""
SELECT *
FROM bronze_mongodb_orders_flat
""").show(truncate=False)

StatementMeta(, b12ffaba-d39d-478c-a983-946771fd6bbd, 23, Finished, Available, Finished, False)

+------------------------+--------+-----------+-------------+--------------+----------------------------------------------------------------------------------------------------------------------------------------------------------+----------+--------------+--------------+--------------+------------+-------------------+
|mongo_id                |order_id|customer_id|customer_name|customer_email|products                                                                                                                                                  |payment_id|payment_method|payment_status|payment_amount|order_status|order_time         |
+------------------------+--------+-----------+-------------+--------------+----------------------------------------------------------------------------------------------------------------------------------------------------------+----------+--------------+--------------+--------------+------------+-------------------+
|69de5c58d5c58a217003c92e|1001    |30

In [ ]:
# ----------------------------------------------------------
# DOCUMENTATION
# ----------------------------------------------------------
# Authentication challenge summary:
#
# 1. Azure Key Vault was created and configured for MongoDB secrets.
# 2. DefaultAzureCredential() could not be used because Fabric runtime
#    did not expose a usable Azure authentication source.
# 3. Direct Key Vault secret retrieval also failed because of tenant /
#    issuer mismatch between Fabric runtime identity and the Key Vault tenant.
# 4. Therefore, Fabric Variable Library was selected as the runtime-safe
#    configuration mechanism for notebook execution.
#
# Additional connectivity challenge:
# MongoDB Atlas network access initially blocked Fabric runtime connectivity.
# This was resolved during development by adding 0.0.0.0/0 temporarily to
# the Atlas IP Access List.
#
# Final execution approach:
# Fabric Variable Library
# → NotebookUtils
# → MongoDB Atlas
# → Spark DataFrame
# → Bronze Delta tables

print("MongoDB ingestion notebook completed successfully.")